# Effect of Antisemitic Tropes on Recall 

In [1]:
import pandas as pd
import numpy as np
import os
import sys
from os.path import join, abspath

from os.path import join
sys.path.append("../..")

from config import DATA_DIR
from utils.classification_helpers import group_ihra_content, group_lexicon_content
from utils.stats_helpers import pairwise_segment_tests
from utils.analysis_helpers import calculate_recall, summarize_recall_stats, group_dfs_by_row_index_pivot, explode_df, test_recall_per_content_group_compared_to_base, combine_dataset_tests

pd.set_option("display.max_rows", 100)

PROVIDER = "gemini"

In [2]:
bloomington_tmp = pd.read_feather(join(DATA_DIR, f"{PROVIDER}_bloomington_label_1.feather"))
decoding = pd.read_feather(join(DATA_DIR, f"{PROVIDER}_decoding_label_1.feather"))

In [3]:
print("Bloomington columns:", bloomington_tmp.columns)

Bloomington columns: Index(['comment_cleaned', 'id', 'classification_tax', 'explanation_tax',
       'classification_tax_ex', 'explanation_tax_ex', 'keyword',
       'ihra_section_1', 'ihra_section_2', 'ihra_sections',
       'classification_lexicon', 'explanation_lexicon',
       'classification_ihra_binary', 'classification_ihra_binary_cleaned',
       'classification_ihra_explanation',
       'classification_ihra_explanation_cleaned',
       'explanation_ihra_explanation', 'classification_no_kb',
       'classification_no_kb_cleaned', 'explanation_ihra_explanation_sections',
       'explanation_lexicon_chapters', 'explanation_lexicon_chapters_no',
       'explanation_lexicon_sections', 'explanation_tax_chapters',
       'explanation_tax_ex_chapters', 'explanation_tax_chapters_no',
       'explanation_tax_ex_chapters_no', 'explanation_tax_sections',
       'explanation_tax_ex_sections'],
      dtype='object')


In [7]:
column_name_renaming = {
    'classification_no_kb_cleaned': 'NO_KB',
    'classification_ihra_explanation_cleaned': 'IHRA',
    'classification_tax': 'TAX',
    'classification_tax_ex': 'TAX_EX',
    'classification_lexicon': 'LEXICON',
}

bloomington_tmp.rename(columns=column_name_renaming, inplace=True)
decoding.rename(columns=column_name_renaming, inplace=True)

REL_CLASS_COLS = column_name_renaming.values()

In [8]:
# Create a new df for Bloomington data with a new column "ihra_sec_new" containing both annotators' decisions.
bloomington_copy_x = bloomington_tmp.copy()
bloomington_copy_y = bloomington_tmp.copy()
bloomington_copy_x["ihra_sec_new"] = bloomington_tmp["ihra_section_1"]
bloomington_copy_y["ihra_sec_new"] = bloomington_tmp["ihra_section_2"]
bloomington = pd.concat([bloomington_copy_x, bloomington_copy_y], ignore_index=True)
print(len(bloomington))
bloomington = bloomington[bloomington["ihra_sec_new"]!=13] # there are a few errors coded as 13
print(len(bloomington))

3716
3635


## Bloomington

#### a. Within each KB

In [9]:

class_cols_to_recalls = {}
for classification_column in REL_CLASS_COLS:
    recall,support, correct = calculate_recall(df=bloomington, classification_column=classification_column, split_by='keyword', two_annotators=True)
    recall_summary = summarize_recall_stats(recall, support, correct)
    print(f"------{classification_column}-------")
    print(recall_summary)
    class_cols_to_recalls[classification_column]=recall_summary
    pairwise = pairwise_segment_tests(recall_summary, correction="holm")
    print("\n[Pairwise tests vs each other] (Holm-corrected)")
    print(pairwise[["seg_A","seg_B","p_raw","p_adj","significant", "effect_h"]])
    print("---------\n")    


------NO_KB-------
         Recall  Support  Correct  Missed
Israel     0.32      641      203     438
Kikes      0.97       89       86       3
ZioNazi    0.99      397      392       5
Jews       0.71      689      488     201

[Pairwise tests vs each other] (Holm-corrected)
     seg_A    seg_B         p_raw         p_adj  significant  effect_h
1   Israel  ZioNazi  1.890048e-99  1.134029e-98         True -1.738729
2   Israel     Jews  6.202992e-46  3.101496e-45         True -0.801713
0   Israel    Kikes  3.007951e-31  1.203181e-30         True -1.590898
5  ZioNazi     Jews  3.288038e-29  9.864115e-29         True  0.937016
4    Kikes     Jews  3.774163e-07  7.548326e-07         True  0.789185
3    Kikes  ZioNazi  1.656457e-01  1.656457e-01        False -0.147831
---------

------IHRA-------
         Recall  Support  Correct  Missed
Israel     0.38      641      246     395
Kikes      0.97       89       86       3
ZioNazi    0.99      397      394       3
Jews       0.73      689    

#### b. Between KBs

In [10]:
grouped = group_dfs_by_row_index_pivot(class_cols_to_recalls)
print(grouped["Kikes"])

         Correct  Missed  Recall  Support
dataset                                  
IHRA          86       3    0.97       89
LEXICON       86       3    0.97       89
NO_KB         86       3    0.97       89
TAX           87       2    0.98       89
TAX_EX        89       0    1.00       89


In [11]:
print(grouped["Israel"])

         Correct  Missed  Recall  Support
dataset                                  
IHRA         246     395    0.38      641
LEXICON      500     141    0.78      641
NO_KB        203     438    0.32      641
TAX          541     100    0.84      641
TAX_EX       481     160    0.75      641


In [14]:
# Pairwise chi-squared (auto-fallback to Fisher), Holm-corrected
model_comparison_per_group = pd.DataFrame(columns=["seg_A","seg_B", "p_raw","p_adj","significant", "effect_h"])

index = []
for i, k in enumerate(grouped.keys()):
    pairwise = pairwise_segment_tests(grouped[k], correction="holm")
    model_comparison_per_group.loc[i] = pairwise.loc[1] # change 1 to another number to get a different comparison
    index.append(k)
model_comparison_per_group["keyword"] = index
model_comparison_per_group.set_index("keyword", inplace=True)
model_comparison_per_group

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
keyword,,,,,,
Israel,IHRA,NO_KB,0.013935,0.02787,True,0.125902
Jews,IHRA,NO_KB,0.401363,0.802725,False,0.04455
Kikes,IHRA,NO_KB,1.0,1.0,False,0.0
ZioNazi,IHRA,NO_KB,0.725175,1.0,False,0.0


### Evaluation by content groups

Content groups allow to compare IHRA sections with LEXICON chapters

In [15]:
bloomington["ihra_content"] = bloomington["ihra_sec_new"].map(group_ihra_content)
print(bloomington["ihra_content"].value_counts(dropna=False, normalize=True))
print(type(bloomington["ihra_content"][0]))

ihra_content
israel                  0.576891
classic_power           0.279230
aggressive              0.096286
second_postholocaust    0.047593
Name: proportion, dtype: float64
<class 'str'>


#### a. Within each KB

In [16]:
class_cols_to_recalls = {}
for classification_column in REL_CLASS_COLS:
    recall,support, correct = calculate_recall(df=bloomington, classification_column=classification_column, split_by='ihra_content', two_annotators=True)
    recall_summary = summarize_recall_stats(recall, support, correct)
    print(f"--------{classification_column}--------")
    print(recall_summary)
    class_cols_to_recalls[classification_column]=recall_summary
    pairwise = pairwise_segment_tests(recall_summary, correction="holm")
    print(pairwise[["seg_A","seg_B","p_raw","p_adj","significant", "effect_h"]])
    print("---------\n")    

--------NO_KB--------
                      Recall  Support  Correct  Missed
israel                  0.51     1048      536     512
classic_power           0.85      507      429      78
aggressive              0.82      175      144      31
second_postholocaust    0.71       86       61      25
           seg_A                 seg_B         p_raw         p_adj  \
0         israel         classic_power  6.338866e-37  3.803320e-36   
1         israel            aggressive  3.126634e-14  1.563317e-13   
2         israel  second_postholocaust  6.254354e-04  2.501742e-03   
4  classic_power  second_postholocaust  3.244186e-03  9.732559e-03   
5     aggressive  second_postholocaust  5.236671e-02  1.047334e-01   
3  classic_power            aggressive  5.448419e-01  5.448419e-01   

   significant  effect_h  
0         True -0.755396  
1         True -0.674497  
2         True -0.413444  
4         True  0.341952  
5        False  0.261053  
3        False  0.080899  
---------

--------IHRA

In [17]:
kb_to_recall_b = {}
for kb in REL_CLASS_COLS:
    kb_to_recall_b[kb] = class_cols_to_recalls[kb]['Recall'].to_dict()
pd.DataFrame(kb_to_recall_b)

,NO_KB,IHRA,TAX,TAX_EX,LEXICON
israel,0.51,0.56,0.87,0.79,0.82
classic_power,0.85,0.86,0.90,0.91,0.89
aggressive,0.82,0.87,0.92,0.91,0.90
second_postholocaust,0.71,0.64,0.83,0.79,0.84


#### b. Between KBs

In [35]:
recall_tests_b = pd.concat([test_recall_per_content_group_compared_to_base(class_cols_to_recalls, i) for i in [0,4,8]])
recall_tests_b

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
content_group,,,,,,
aggressive,NO_KB,IHRA,0.318632,1.0,False,0.094225
classic_power,NO_KB,IHRA,0.222034,1.0,False,0.090863
israel,NO_KB,IHRA,0.0,0.0,True,0.574208
second_postholocaust,NO_KB,IHRA,0.005508,0.055079,False,0.463969
aggressive,LEXICON,NO_KB,0.043382,0.34706,False,0.232797
classic_power,LEXICON,NO_KB,0.079967,0.479802,False,0.119268
israel,LEXICON,NO_KB,0.0,0.0,True,0.674497
second_postholocaust,LEXICON,NO_KB,0.068609,0.480265,False,0.314317
aggressive,NO_KB,TAX_EX,0.017641,0.176412,False,-0.266913


## Decoding

### Evaluation by content groups



In [36]:
decoding["lexicon_chapter_groups"] = decoding["comment_codes_all_chapters"].map(group_lexicon_content)

In [37]:
# restrict to texts with maximum two content groups assigned
decoding = decoding[decoding['lexicon_chapter_groups'].map(lambda x: len(x) <=2 and len(x)>0)].copy()
decoding["lexicon_chapter_groups"] = decoding["lexicon_chapter_groups"].map(lambda x: x if len(x)==2 else x*2)
decoding = explode_df(decoding, "lexicon_chapter_groups")

In [38]:
decoding["lexicon_chapter_groups"].value_counts()

lexicon_chapter_groups
israel                  2667
classic_power           2169
second_postholocaust     775
aggressive               131
Name: count, dtype: int64

#### a. Within each KB

In [39]:
for classification_column in REL_CLASS_COLS:
    recall,support, correct = calculate_recall(df=decoding, classification_column=classification_column, split_by='lexicon_chapter_groups', two_annotators=True)
    recall_summary = summarize_recall_stats(recall, support, correct)
    print(f"--------{classification_column}--------")
    print(recall_summary)
    class_cols_to_recalls[classification_column]=recall_summary
    pairwise = pairwise_segment_tests(recall_summary, correction="holm")
    print("\n[Pairwise tests vs each other] (Holm-corrected)")
    print(pairwise[["seg_A","seg_B","p_raw","p_adj","significant", "effect_h"]])
    print("---------\n")   

--------NO_KB--------
                      Recall  Support  Correct  Missed
israel                  0.41     1333      544     789
classic_power           0.51     1084      554     530
second_postholocaust    0.39      387      151     236
aggressive              0.75       65       49      16

[Pairwise tests vs each other] (Holm-corrected)
                  seg_A                 seg_B         p_raw         p_adj  \
2                israel            aggressive  7.490258e-08  4.494155e-07   
5  second_postholocaust            aggressive  9.970463e-08  4.985231e-07   
0                israel         classic_power  5.292447e-07  2.116979e-06   
3         classic_power  second_postholocaust  5.642880e-05  1.692864e-04   
4         classic_power            aggressive  2.340244e-04  4.680488e-04   
1                israel  second_postholocaust  5.662088e-01  5.662088e-01   

   significant  effect_h  
2         True -0.704585  
5         True -0.745413  
0         True -0.200988  
3     

In [40]:
kb_to_recall_d = {}
for kb in REL_CLASS_COLS:
    kb_to_recall_d[kb] = class_cols_to_recalls[kb]['Recall'].to_dict()
pd.DataFrame(kb_to_recall_d)

,NO_KB,IHRA,TAX,TAX_EX,LEXICON
israel,0.41,0.46,0.93,0.85,0.87
classic_power,0.51,0.50,0.82,0.80,0.85
second_postholocaust,0.39,0.36,0.76,0.65,0.77
aggressive,0.75,0.78,0.89,0.89,0.88


#### b. Between KBs

In [41]:
recall_tests_d = pd.concat([test_recall_per_content_group_compared_to_base(class_cols_to_recalls, i) for i in [0,4,8]])
recall_tests_d

,seg_A,seg_B,p_raw,p_adj,significant,effect_h
content_group,,,,,,
aggressive,NO_KB,IHRA,0.242182,1.0,False,0.268927
classic_power,NO_KB,IHRA,0.0,0.0,True,0.775397
israel,NO_KB,IHRA,0.0,0.0,True,0.913156
second_postholocaust,NO_KB,IHRA,0.0,0.0,True,0.854231
aggressive,LEXICON,NO_KB,0.113563,0.9085,False,0.339714
classic_power,LEXICON,NO_KB,0.0,0.0,True,0.755396
israel,LEXICON,NO_KB,0.0,0.0,True,1.014057
second_postholocaust,LEXICON,NO_KB,0.0,0.0,True,0.792252
aggressive,NO_KB,TAX_EX,0.065962,0.659625,False,-0.371067


In [26]:
print(recall_tests_d)

                      seg_A   seg_B     p_raw     p_adj significant  effect_h
content_group                                                                
aggressive            NO_KB    IHRA  0.835099       1.0       False -0.070787
classic_power         NO_KB    IHRA  0.519366  0.519366       False  0.020001
israel                NO_KB    IHRA  0.004917  0.004917        True -0.100901
second_postholocaust  NO_KB    IHRA  0.504505  0.504505       False   0.06198
aggressive            NO_KB     TAX  0.065962  0.395775       False -0.371067
classic_power         NO_KB     TAX       0.0       0.0        True -0.674497
israel                NO_KB     TAX       0.0       0.0        True -1.216256
second_postholocaust  NO_KB     TAX       0.0       0.0        True -0.768665
aggressive            NO_KB  TAX_EX  0.065962  0.395775       False -0.371067
classic_power         NO_KB  TAX_EX       0.0       0.0        True   -0.6235
israel                NO_KB  TAX_EX       0.0       0.0        T

## Dataset union 

- equal dataset weight
- by content groups

In [42]:
kb_to_recall_bd = {}
for classification_column in REL_CLASS_COLS:
    tmp_b = kb_to_recall_b[classification_column]
    tmp_d = kb_to_recall_d[classification_column]
    tmp = {k:0.5*(tmp_b[k] + tmp_d[k]) for k in tmp_b.keys()}
    kb_to_recall_bd[classification_column] = tmp
       

In [43]:
kb_to_recall_bd = pd.DataFrame(kb_to_recall_bd)
kb_to_recall_bd

,NO_KB,IHRA,TAX,TAX_EX,LEXICON
israel,0.460,0.510,0.900,0.820,0.845
classic_power,0.680,0.680,0.860,0.855,0.870
aggressive,0.785,0.825,0.905,0.900,0.890
second_postholocaust,0.550,0.500,0.795,0.720,0.805


In [44]:
kb_to_recall_bd.loc["mean",:] = kb_to_recall_bd.mean().tolist()
kb_to_recall_bd

,NO_KB,IHRA,TAX,TAX_EX,LEXICON
israel,0.46000,0.51000,0.900,0.82000,0.8450
classic_power,0.68000,0.68000,0.860,0.85500,0.8700
aggressive,0.78500,0.82500,0.905,0.90000,0.8900
second_postholocaust,0.55000,0.50000,0.795,0.72000,0.8050
mean,0.61875,0.62875,0.865,0.82375,0.8525


In [45]:
kb_to_recall_bd.to_parquet(f"{PROVIDER}_recall_by_trope.parquet", index=True)


In [46]:
combined = combine_dataset_tests(
    recall_tests_b,
    recall_tests_d,
)

In [47]:
combined

,content_group,seg_A,seg_B,p_raw,effect_h,p_adj,significant
2,israel,NO_KB,IHRA,7.274069e-30,0.743682,8.728883e-29,True
6,israel,LEXICON,NO_KB,7.274069e-30,0.844277,8.728883e-29,True
10,israel,NO_KB,TAX_EX,7.274069e-30,-0.777556,8.728883e-29,True
9,classic_power,NO_KB,TAX_EX,7.425889e-15,-0.404757,6.683300e-14,True
3,second_postholocaust,NO_KB,IHRA,2.196326e-14,0.659100,1.757061e-13,True
7,second_postholocaust,LEXICON,NO_KB,3.319768e-12,0.553284,2.323838e-11,True
5,classic_power,LEXICON,NO_KB,4.714943e-12,0.437332,2.828966e-11,True
1,classic_power,NO_KB,IHRA,6.179881e-11,0.433130,3.089941e-10,True
11,second_postholocaust,NO_KB,TAX_EX,7.323854e-09,-0.355895,2.929542e-08,True
8,aggressive,NO_KB,TAX_EX,2.899918e-03,-0.318990,8.699755e-03,True
